# DX 704 Week 6 Project

This project will develop a treatment plan for a fictious illness "Twizzleflu".
Twizzleflu is a mild illness caused by a virus.
The main symptoms are a mild fever, fidgeting, and kicking the blankets off the bed or couch.
Mild dehydration has also been reported in more severe cases.
These symptoms typically last 1-2 weeks without treatment.
Word on the internet says that Twizzleflu can be cured faster by drinking copious orange juice, but this has not been supported by evidence so far.
You will be provided with a theoretical model of Twizzleflu modeled as a Markov decision process.
Based on the model, you will compute optimal treatment plans to optimize different criteria, and compare patient discomfort with the different plans.

The full project description, a template notebook, and raw data are available on GitHub: [Project 6 Materials](https://github.com/bu-cds-dx704/dx704-project-06).

We will model Twizzleflu as a Markov decision process.
The model transition probabilities are provided in the file "twizzleflu-transitions.tsv" and the expected rewards are in "twizzleflu-rewards.tsv".
The goal for Twizzleflu is to minimize the expected discomfort of the patient which is expressed as negative rewards in the file.

## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Evaluate a Do Nothing Plan

One of the treatment actions is to do nothing.
Calculate the expected discomfort (not rewards) of a policy that always does nothing.

Hint: for this value calculation and later ones, use value iteration.
The analytical solution has difficulties in practice when there is no discount factor.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def compute_qT_once(R, P, gamma, v):
    return R + gamma * P @ v

def iterate_values_once(R, P, gamma, v):
    return np.max(compute_qT_once(R, P, gamma, v), axis=0)

def value_iteration(R, P, gamma, max_iterations=100, tolerance=0.001):
    # initial approximation v_0
    v_old = np.zeros(R.shape[-1])

    for i in range(max_iterations):
        # compute v_{i+1}
        v_new = iterate_values_once(R, P, gamma, v_old)

        # check if values did not change much
        if np.max(np.abs(v_new - v_old)) < tolerance:
            return v_new

        v_old = v_new

    # return v_{max_iterations}
    return v_old

In [2]:
R = pd.read_csv('twizzleflu-rewards.tsv', sep='\t')
R

,action,state,reward
0,do-nothing,exposed-1,0.000
1,do-nothing,exposed-2,0.000
2,do-nothing,exposed-3,0.000
3,do-nothing,symptoms-1,-0.500
4,do-nothing,symptoms-2,-1.000
5,do-nothing,symptoms-3,-0.500
6,do-nothing,recovered,0.000
7,drink-oj,exposed-1,0.000
8,drink-oj,exposed-2,0.000
9,drink-oj,exposed-3,0.000


In [3]:
P = pd.read_csv('twizzleflu-transitions.tsv', sep='\t')
P

,action,state,next_state,probability
0,do-nothing,exposed-1,exposed-2,0.80
1,do-nothing,exposed-1,recovered,0.20
2,do-nothing,exposed-2,exposed-3,0.80
3,do-nothing,exposed-2,recovered,0.20
4,do-nothing,exposed-3,symptoms-1,0.80
5,do-nothing,exposed-3,recovered,0.20
6,do-nothing,symptoms-1,symptoms-1,0.70
7,do-nothing,symptoms-1,symptoms-2,0.30
8,do-nothing,symptoms-2,symptoms-2,0.70
9,do-nothing,symptoms-2,symptoms-3,0.30


In [ ]:
states = R['state'].unique()
n_states = len(states)

R_do_nothing = R[R['action'] == 'do-nothing']

# Reward would have negative, but since we measure discomfort the higher value = more discomfort
discomfort = -R_do_nothing['reward'].to_numpy()

R_discomfort = discomfort.reshape(1, n_states)

transitions_do_nothing = P[P['action'] == 'do-nothing']
transitions_do_nothing = transitions_do_nothing.pivot(index='state', columns='next_state', values='probability')

P_do_nothing = transitions_do_nothing.reindex(index=states, columns=states).fillna(0.0).to_numpy()
P_do_nothing = P_do_nothing.reshape(1, n_states, n_states)

v_discomfort = value_iteration(R_discomfort, P_do_nothing, gamma=1.0)

results = pd.DataFrame(
    {'state': states, 'expected_discomfort': v_discomfort}
)
results

,state,expected_discomfort
0,exposed-1,3.410978
1,exposed-2,4.264491
2,exposed-3,5.331327
3,symptoms-1,6.664819
4,symptoms-2,4.999779
5,symptoms-3,1.666654
6,recovered,0.000000


Save the expected discomfort by state to a file "do-nothing-discomfort.tsv" with columns state and expected_discomfort.

In [5]:
results.to_csv('do-nothing-discomfort.tsv', sep='\t', index=False)

Submit "do-nothing-discomfort.tsv" in Gradescope.

## Part 2: Compute an Optimal Treatment Plan

Compute an optimal treatment plan for Twizzleflu.
It should minimize the expected discomfort (maximize the rewards).

In [9]:
states = R['state'].unique()
n_states = len(states)

actions = R['action'].unique()
n_actions = len(actions)

R_pivot = R.pivot(index='action', columns='state', values='reward')
R_numpy = R_pivot.reindex(index=actions, columns=states).to_numpy()

P_list = []
for a in actions:
    P_a = (
        P[P['action'] == a]
        .pivot(index='state', columns='next_state', values='probability')
        .reindex(index=states, columns=states)
        .fillna(0.0)
        .to_numpy()
    )
    P_list.append(P_a)

P_numpy= np.array(P_list)

v_reward = value_iteration(R_numpy, P_numpy, gamma=1.0)

Q = compute_qT_once(R_numpy, P_numpy, gamma=1.0, v=v_reward)

best_actions_indices = np.argmax(Q, axis=0)
best_actions = [actions[i] for i in best_actions_indices]

optimal_treatment = pd.DataFrame(
    {'state': states, 'action': best_actions}
)
optimal_treatment

,state,action
0,exposed-1,sleep-8
1,exposed-2,sleep-8
2,exposed-3,sleep-8
3,symptoms-1,drink-oj
4,symptoms-2,drink-oj
5,symptoms-3,drink-oj
6,recovered,do-nothing


Save the optimal actions for each state to a file "minimum-discomfort-actions.tsv" with columns state and action.

In [10]:
optimal_treatment.to_csv('minimum-discomfort-actions.tsv', sep='\t', index=False)

Submit "minimum-discomfort-actions.tsv" in Gradescope.

## Part 3: Expected Discomfort

Using your previous optimal policy, compute the expected discomfort for each state.

In [25]:
states = optimal_treatment['state'].unique()
n_states = len(states)


treatment_pairs = pd.MultiIndex.from_frame(optimal_treatment[['state', 'action']])
R_filtered = R.set_index(['state', 'action']).loc[treatment_pairs].reset_index()

# Reward would have negative, but since we measure discomfort the higher value = more discomfort
R_discomfort = -R_filtered['reward'].to_numpy()

transitions_plans = pd.MultiIndex.from_frame(optimal_treatment[['action', 'state']])
transitions_filtered = P.set_index(['action', 'state']).loc[transitions_plans].reset_index()
transitions_pivot = transitions_filtered.pivot(index='state', columns='next_state', values='probability')

P_treatment_plans = transitions_pivot.reindex(index=states, columns=states).fillna(0.0).to_numpy()
P_plans_reshaped = P_treatment_plans.reshape(1, n_states, n_states)

v_discomfort = value_iteration(R_discomfort, P_plans_reshaped, gamma=1.0)

treatment_plan_discomfort = pd.DataFrame(
    {'state': states, 'expected_discomfort': v_discomfort}
)
treatment_plan_discomfort

,state,expected_discomfort
0,exposed-1,0.749165
1,exposed-2,1.498689
2,exposed-3,2.997944
3,symptoms-1,5.996779
4,symptoms-2,4.499580
5,symptoms-3,1.499973
6,recovered,0.000000


Save your results in a file "minimum-discomfort-values.tsv" with columns state and expected_discomfort.

In [26]:
treatment_plan_discomfort.to_csv('minimum-discomfort-values.tsv', sep='\t', index=False)

Submit "minimum-discomfort-values.tsv" in Gradescope.

## Part 4: Minimizing Twizzleflu Duration

Modifiy the Markov decision process to minimize the days until the Twizzle flu is over.
To do so, change the reward function to always be -1 if the current state corresponds to being sick (must have symptoms, exposed does not count) and 0 otherwise.
To be clear, the action does not matter for this reward function.


In [27]:
R

,action,state,reward
0,do-nothing,exposed-1,0.000
1,do-nothing,exposed-2,0.000
2,do-nothing,exposed-3,0.000
3,do-nothing,symptoms-1,-0.500
4,do-nothing,symptoms-2,-1.000
5,do-nothing,symptoms-3,-0.500
6,do-nothing,recovered,0.000
7,drink-oj,exposed-1,0.000
8,drink-oj,exposed-2,0.000
9,drink-oj,exposed-3,0.000


In [40]:
duration_rewards = R.copy()

symptoms = [i for i in duration_rewards['state'].unique() if i.startswith('symptoms')]
other_states = duration_rewards[~duration_rewards['state'].isin(symptoms)]['state']

duration_rewards.loc[duration_rewards['state'].isin(symptoms), 'reward'] = -1
duration_rewards.loc[duration_rewards['state'].isin(other_states), 'reward'] = 0

duration_rewards

,action,state,reward
0,do-nothing,exposed-1,0.0
1,do-nothing,exposed-2,0.0
2,do-nothing,exposed-3,0.0
3,do-nothing,symptoms-1,-1.0
4,do-nothing,symptoms-2,-1.0
5,do-nothing,symptoms-3,-1.0
6,do-nothing,recovered,0.0
7,drink-oj,exposed-1,0.0
8,drink-oj,exposed-2,0.0
9,drink-oj,exposed-3,0.0


Save your new reward function in a file "duration-rewards.tsv" in the same format as "twizzleflu-rewards.tsv".

In [41]:
duration_rewards.to_csv('duration-rewards.tsv', sep='\t', index=False)

Submit "duration-rewards.tsv" in Gradescope.

## Part 5: Optimize for Shorter Twizzleflu

Compute an optimal policy to minimize the duration of Twizzleflu.

In [43]:
states = duration_rewards['state'].unique()
n_states = len(states)

actions = duration_rewards['action'].unique()
n_actions = len(actions)

R_pivot = duration_rewards.pivot(index='action', columns='state', values='reward')
R_numpy = R_pivot.reindex(index=actions, columns=states).to_numpy()

P_list = []
for a in actions:
    P_a = (
        P[P['action'] == a]
        .pivot(index='state', columns='next_state', values='probability')
        .reindex(index=states, columns=states)
        .fillna(0.0)
        .to_numpy()
    )
    P_list.append(P_a)

P_numpy= np.array(P_list)

v_reward = value_iteration(R_numpy, P_numpy, gamma=1.0)

Q = compute_qT_once(R_numpy, P_numpy, gamma=1.0, v=v_reward)

best_actions_indices = np.argmax(Q, axis=0)
best_actions = [actions[i] for i in best_actions_indices]

shorter_twizzleflu = pd.DataFrame(
    {'state': states, 'action': best_actions}
)
shorter_twizzleflu

,state,action
0,exposed-1,sleep-8
1,exposed-2,sleep-8
2,exposed-3,sleep-8
3,symptoms-1,do-nothing
4,symptoms-2,do-nothing
5,symptoms-3,do-nothing
6,recovered,do-nothing


Save the optimal actions for each state to a file "minimum-duration-actions.tsv" with columns state and action.

In [44]:
shorter_twizzleflu.to_csv('minimum-duration-actions.tsv', sep='\t', index=False)

Submit "minimum-duration-actions.tsv" in Gradescope.

## Part 6: Shorter Twizzleflu?

Compute the expected number of days sick for each state to a file.

In [45]:
states = shorter_twizzleflu['state'].unique()
n_states = len(states)

shorter_actions = pd.MultiIndex.from_frame(shorter_twizzleflu[['state', 'action']])
R_filtered = R.set_index(['state', 'action']).loc[shorter_actions].reset_index()

R_plan = R_filtered['reward'].to_numpy()

transitions_plans = pd.MultiIndex.from_frame(shorter_twizzleflu[['action', 'state']])
transitions_filtered = P.set_index(['action', 'state']).loc[transitions_plans].reset_index()
transitions_pivot = transitions_filtered.pivot(index='state', columns='next_state', values='probability')

P_shorter_plans = transitions_pivot.reindex(index=states, columns=states).fillna(0.0).to_numpy()
P_shorter_plans_reshaped = P_shorter_plans.reshape(1, n_states, n_states)

v_discomfort = value_iteration(R_plan, P_shorter_plans_reshaped, gamma=1.0)

treatment_plan_discomfort = pd.DataFrame(
    {'state': states, 'expected_sick_days': v_discomfort}
)
treatment_plan_discomfort

,state,expected_sick_days
0,exposed-1,-0.832556
1,exposed-2,-1.665517
2,exposed-3,-3.331634
3,symptoms-1,-6.664159
4,symptoms-2,-4.999692
5,symptoms-3,-1.666648
6,recovered,0.000000


Save the expected sick days for each state to a file "minimum-duration-days.tsv" with columns state and expected_sick_days.

In [46]:
treatment_plan_discomfort.to_csv('minimum-duration-days.tsv', sep='\t', index=False)

Submit "minimum-duration-days.tsv" in Gradescope.

## Part 7: Speed vs Pampering

Compute the expected discomfort using the policy to minimize days sick, and compare the results to the expected discomfort when optimizing to minimize discomfort.

Hint: Make sure that you are reporting on expected discomfort, not repeating the expected number of days.

In [ ]:
# YOUR CHANGES HERE

...

Save the results to a file "policy-comparison.tsv" with columns state, speed_discomfort, and minimize_discomfort.

In [ ]:
# YOUR CHANGES HERE

...

Submit "policy-comparison.tsv" in Gradescope.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.

## Part 9: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.